<a href="https://colab.research.google.com/github/FrogFart/iPhone-Backup-Analyzer-2/blob/master/Colab_ComfyUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# System & Python dependencies
!apt -y update -qq
!apt -y install -qq libgl1-mesa-glx wget git

# Clone ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI

# Install Python packages
!pip install -r requirements.txt

import torch
print("✅ CUDA available:", torch.cuda.is_available())
print("💡 GPU device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


64 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Package libgl1-mesa-glx is not available, but is referred to by another package.
This may mean that the package is missing, has been obsoleted, or
is only available from another source

E: Package 'libgl1-mesa-glx' has no installation candidate
Cloning into 'ComfyUI'...
remote: Enumerating objects: 59042, done.
remote: Counting objects: 100% (419/419), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 59042 (delta 351), reused 278 (delta 251), pack-reused 58623 (from 4)
Receiving objects: 100% (59042/59042), 94.30 MiB | 53.47 MiB/s, done.
Resolving deltas: 100% (40670/40670), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 127.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# 📂 Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Define your base folder inside Google Drive
GDRIVE_BASE = '/content/drive/MyDrive/ComfyUI'

# 📁 Define required subfolders
REQUIRED_FOLDERS = [
    'models/checkpoints',
    'models/controlnet',
    'models/vae',
    'models/upscale_models',
    'models/clip',
    'models/unet',
    'custom_nodes',
    'input',
    'output',
    'temp',
    'user'
]

# ✅ Create folders if not exist
for folder in REQUIRED_FOLDERS:
    full_path = os.path.join(GDRIVE_BASE, folder)
    os.makedirs(full_path, exist_ok=True)

print("✅ All required folders are verified or created in Google Drive.")

Mounted at /content/drive
✅ All required folders are verified or created in Google Drive.


In [3]:
import os

COMFYUI_PATH = '/content/ComfyUI'
GDRIVE_BASE = '/content/drive/MyDrive/ComfyUI'

LINKS = {
    'models': f'{GDRIVE_BASE}/models',
    'custom_nodes': f'{GDRIVE_BASE}/custom_nodes',
    'input': f'{GDRIVE_BASE}/input',
    'output': f'{GDRIVE_BASE}/output',
    'temp': f'{GDRIVE_BASE}/temp',
    'user': f'{GDRIVE_BASE}/user'
}

for name, target in LINKS.items():
    source = os.path.join(COMFYUI_PATH, name)
    # Remove old local folder
    if os.path.islink(source) or os.path.isdir(source):
        !rm -rf "{source}"
    # Create symlink
    os.symlink(target, source)

print("✅ ComfyUI is now fully configured to use Google Drive.")

✅ ComfyUI is now fully configured to use Google Drive.


In [4]:
import os

# ✅ CONFIG
drive_model_path = "/content/drive/MyDrive/ComfyUI/models/checkpoints"
model_filename = "sd_xl_base_1.0.safetensors"
model_url = "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors"
force_refresh = False  # 🔁 Set to True to redownload even if file exists

# ✅ Construct full path
model_path = os.path.join(drive_model_path, model_filename)

# ✅ Ensure model folder exists
os.makedirs(drive_model_path, exist_ok=True)

# ✅ Logic to download or refresh
if not os.path.exists(model_path):
    print("⬇️  Model not found in Google Drive. Downloading...")
    !wget -O "{model_path}" "{model_url}"
    print("✅ Model downloaded to Google Drive.")
elif force_refresh:
    print("🔁 Force refresh is enabled. Re-downloading model...")
    !wget -O "{model_path}" "{model_url}"
    print("✅ Model refreshed in Google Drive.")
else:
    print("✅ Model already exists in Google Drive. Skipping download.")


✅ Model already exists in Google Drive. Skipping download.


In [5]:
!rm -f ngrok ngrok.zip ngrok-stable-linux-amd64.tgz
!wget -O ngrok-stable-linux-amd64.tgz https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xvzf ngrok-stable-linux-amd64.tgz
!chmod +x ngrok
!./ngrok version
!./ngrok authtoken 3KYZBIGWf21lByIl6GyMsoJfwOs_4kfMw5v9TTYNttTnu1Jft

--2026-10-12 05:44:06--  https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
Resolving bin.equinox.io (bin.equinox.io)... 35.71.179.82, 13.248.244.96, 99.83.220.108, ...
Connecting to bin.equinox.io (bin.equinox.io)|35.71.179.82|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12104579 (12M) [application/octet-stream]
Saving to: ‘ngrok-stable-linux-amd64.tgz’

ngrok-stable-linux- 100%[===================>]  11.54M  --.-KB/s    in 0.1s    

2026-10-12 05:44:07 (94.6 MB/s) - ‘ngrok-stable-linux-amd64.tgz’ saved [12104579/12104579]

ngrok
ngrok version 3.39.11
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [6]:
# ComfyUI Manager
!pip install "numpy==1.26.4" --force-reinstall --quiet > /dev/null 2>&1
!rm -rf /content/ComfyUI/custom_nodes/ComfyUI-Manager
!git clone --quiet https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
print("✅ ComfyUI-Manager installed and numpy version is set for compatibility.")


✅ ComfyUI-Manager installed and numpy version is set for compatibility.


In [11]:
import subprocess
import time
import requests

# Start ngrok tunnel to port 8188
ngrok_process = subprocess.Popen(['./ngrok', 'http', '8188'],
                                 stdout=subprocess.DEVNULL,
                                 stderr=subprocess.STDOUT)

# Wait and retry connection
for i in range(10):
    try:
        r = requests.get('http://localhost:4040/api/tunnels')
        public_url = r.json()['tunnels'][0]['public_url']
        print(f"✅ Ngrok tunnel established: {public_url}")
        break
    except Exception as e:
        print(f"⏳ Attempt {i+1}/10: Ngrok not ready yet...")
        time.sleep(2)
else:
    print("❌ Ngrok failed to start. Please restart the runtime and try again.")



⏳ Attempt 1/10: Ngrok not ready yet...
✅ Ngrok tunnel established: https://saturate-colony-unsmooth.ngrok-free.dev


In [10]:
# Additional Prereqs
!pip install llama-cpp-python
!pip install boto3
!pip install gguf
!pip install onnxruntime
!pip install sageattention
!pip install ftfy
!pip install onnx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 MB 38.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.36-py3-none-linux_x86_64.whl size=25035101 sha256=c24ccebf38aac97df6b8d7a7055c7d673a406a03e8d7f895bdada57a4b3d5349
  Stored in directory: /root/.cache/pip/wheels/e5/bf/ad/f5f50f854633df40f54d89e4e5271e987476036203819f8b4c
Successfully built llama-cpp-python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 188.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [ ]:
# Start ComfyUI with public access
%cd /content/ComfyUI
!python main.py --listen 0.0.0.0 --port 8188   --cuda-device 0


/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Set cuda device to: 0
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-10-12 07:17:07.027
** Platform: Linux
** Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log
[INFO] 
Prestartup times for custom nodes:
[INFO]    2.2 seconds: /content/ComfyUI/custom